In [5]:
import pandas as pd

presentations = pd.read_csv("data/processed/ED/ed_presentations.csv")
demographics = pd.read_csv("data/processed/ED/ed_demographics.csv")
diagnosis_groups = pd.read_csv("data/processed/ED/ed_diagnosis_groups.csv")
top_diagnoses = pd.read_csv("data/processed/ED/ed_top_diagnoses.csv")
waiting_times = pd.read_csv("data/processed/ED/ed_waiting_times.csv")
triage = pd.read_csv("data/processed/ED/ed_triage_performance.csv")
length_of_stay = pd.read_csv("data/processed/ED/ed_length_of_stay.csv")

In [6]:
presentations.head()

,state,measure,year,value
0,NSW,Presentations,2020–21,3068572.0
1,Vic,Presentations,2020–21,1772359.0
2,Qld,Presentations,2020–21,1887381.0
3,WA,Presentations,2020–21,997816.0
4,SA,Presentations,2020–21,580575.0


In [7]:
dim_state = pd.DataFrame({
    "state": sorted(presentations["state"].unique())
})

dim_state["state_id"] = range(1, len(dim_state) + 1)

dim_state = dim_state[["state_id", "state"]]

dim_state

,state_id,state
0,1,ACT
1,2,NSW
2,3,NT
3,4,Qld
4,5,SA
5,6,Tas
6,7,Vic
7,8,WA


In [8]:
dim_year = pd.DataFrame({
    "year": sorted(presentations["year"].unique())
})

dim_year["year_id"] = range(1, len(dim_year) + 1)

dim_year = dim_year[["year_id", "year"]]

dim_year

,year_id,year
0,1,2020–21
1,2,2021–22
2,3,2022–23
3,4,2023–24
4,5,2024–25


In [9]:
metric_values = pd.concat([
    presentations["measure"],
    waiting_times["measure"]
]).dropna().unique()

dim_metrics = pd.DataFrame({
    "metric": sorted(metric_values)
})

dim_metrics["metric_id"] = range(1, len(dim_metrics) + 1)

dim_metrics = dim_metrics[["metric_id", "metric"]]

dim_metrics

,metric_id,metric
0,1,90th percentile waiting time (minutes)
1,2,"Age-standardised rate per 1,000 population"
2,3,Median waiting time (minutes)
3,4,Presentations
4,5,Proportion seen on time (%)


In [10]:
fact_ed_metrics = pd.concat([
    presentations,
    waiting_times
], ignore_index=True)

fact_ed_metrics.head()

,state,measure,year,value
0,NSW,Presentations,2020–21,3068572.0
1,Vic,Presentations,2020–21,1772359.0
2,Qld,Presentations,2020–21,1887381.0
3,WA,Presentations,2020–21,997816.0
4,SA,Presentations,2020–21,580575.0


In [11]:
fact_ed_metrics = fact_ed_metrics.rename(columns={
    "measure": "metric"
})

fact_ed_metrics.head()

,state,metric,year,value
0,NSW,Presentations,2020–21,3068572.0
1,Vic,Presentations,2020–21,1772359.0
2,Qld,Presentations,2020–21,1887381.0
3,WA,Presentations,2020–21,997816.0
4,SA,Presentations,2020–21,580575.0


In [12]:
fact_ed_metrics = fact_ed_metrics.merge(
    dim_state,
    on="state",
    how="left"
)

fact_ed_metrics.head()

,state,metric,year,value,state_id
0,NSW,Presentations,2020–21,3068572.0,2
1,Vic,Presentations,2020–21,1772359.0,7
2,Qld,Presentations,2020–21,1887381.0,4
3,WA,Presentations,2020–21,997816.0,8
4,SA,Presentations,2020–21,580575.0,5


In [13]:
fact_ed_metrics

,state,metric,year,value,state_id
0,NSW,Presentations,2020–21,3.068572e+06,2
1,Vic,Presentations,2020–21,1.772359e+06,7
2,Qld,Presentations,2020–21,1.887381e+06,4
3,WA,Presentations,2020–21,9.978160e+05,8
4,SA,Presentations,2020–21,5.805750e+05,5
...,...,...,...,...,...
195,ACT,90th percentile waiting time (minutes),2024–25,1.060000e+02,1
196,ACT,Proportion seen on time (%),2024–25,6.167240e+01,1
197,NT,Median waiting time (minutes),2024–25,3.700000e+01,3
198,NT,90th percentile waiting time (minutes),2024–25,1.760000e+02,3


In [14]:
fact_ed_metrics = fact_ed_metrics.merge(
    dim_year,
    on="year",
    how="left"
)

fact_ed_metrics.head()

,state,metric,year,value,state_id,year_id
0,NSW,Presentations,2020–21,3068572.0,2,1
1,Vic,Presentations,2020–21,1772359.0,7,1
2,Qld,Presentations,2020–21,1887381.0,4,1
3,WA,Presentations,2020–21,997816.0,8,1
4,SA,Presentations,2020–21,580575.0,5,1


In [15]:
fact_ed_metrics = fact_ed_metrics.merge(
    dim_metrics,
    on="metric",
    how="left"
)

fact_ed_metrics.head()

,state,metric,year,value,state_id,year_id,metric_id
0,NSW,Presentations,2020–21,3068572.0,2,1,4
1,Vic,Presentations,2020–21,1772359.0,7,1,4
2,Qld,Presentations,2020–21,1887381.0,4,1,4
3,WA,Presentations,2020–21,997816.0,8,1,4
4,SA,Presentations,2020–21,580575.0,5,1,4


In [16]:
fact_ed_metrics = fact_ed_metrics[
    ["state_id", "year_id", "metric_id", "value"]
]

fact_ed_metrics.head()

,state_id,year_id,metric_id,value
0,2,1,4,3068572.0
1,7,1,4,1772359.0
2,4,1,4,1887381.0
3,8,1,4,997816.0
4,5,1,4,580575.0


In [17]:
fact_ed_metrics.shape

(200, 4)

In [19]:
dim_peer_group = pd.DataFrame({
    "peer_group": sorted(triage["peer_group"].unique())
})

dim_peer_group["peer_group_id"] = range(1, len(dim_peer_group) + 1)

dim_peer_group = dim_peer_group[["peer_group_id", "peer_group"]]

dim_peer_group

,peer_group_id,peer_group
0,1,All hospitals
1,2,Other hospitals
2,3,Principal referral and womens and children's h...
3,4,Public acute group A hospitals
4,5,Public acute group B hospitals
5,6,Public acute group C hospitals


In [20]:
dim_triage_category = pd.DataFrame({
    "triage_category": sorted(triage["triage_category"].unique())
})

dim_triage_category["triage_category_id"] = range(1, len(dim_triage_category) + 1)

dim_triage_category = dim_triage_category[
    ["triage_category_id", "triage_category"]
]

dim_triage_category

,triage_category_id,triage_category
0,1,Emergency
1,2,Non-urgent
2,3,Resuscitation
3,4,Semi-urgent
4,5,Urgent


In [21]:
fact_ed_triage = triage.merge(
    dim_state,
    on="state",
    how="left"
)

fact_ed_triage.head()

,peer_group,triage_category,state,value,state_id
0,Principal referral and womens and children's h...,Resuscitation,NSW,99.9548,2
1,Principal referral and womens and children's h...,Emergency,NSW,74.9598,2
2,Principal referral and womens and children's h...,Urgent,NSW,67.904,2
3,Principal referral and womens and children's h...,Semi-urgent,NSW,73.1714,2
4,Principal referral and womens and children's h...,Non-urgent,NSW,86.9421,2


In [22]:
fact_ed_triage = fact_ed_triage.merge(
    dim_peer_group,
    on="peer_group",
    how="left"
)

fact_ed_triage.head()

,peer_group,triage_category,state,value,state_id,peer_group_id
0,Principal referral and womens and children's h...,Resuscitation,NSW,99.9548,2,3
1,Principal referral and womens and children's h...,Emergency,NSW,74.9598,2,3
2,Principal referral and womens and children's h...,Urgent,NSW,67.904,2,3
3,Principal referral and womens and children's h...,Semi-urgent,NSW,73.1714,2,3
4,Principal referral and womens and children's h...,Non-urgent,NSW,86.9421,2,3


In [23]:
fact_ed_triage = fact_ed_triage.merge(
    dim_peer_group,
    on="peer_group",
    how="left"
)

fact_ed_triage.head()

,peer_group,triage_category,state,value,state_id,peer_group_id_x,peer_group_id_y
0,Principal referral and womens and children's h...,Resuscitation,NSW,99.9548,2,3,3
1,Principal referral and womens and children's h...,Emergency,NSW,74.9598,2,3,3
2,Principal referral and womens and children's h...,Urgent,NSW,67.904,2,3,3
3,Principal referral and womens and children's h...,Semi-urgent,NSW,73.1714,2,3,3
4,Principal referral and womens and children's h...,Non-urgent,NSW,86.9421,2,3,3


In [24]:
fact_ed_triage[["peer_group_id_x","peer_group_id_y"]].head()

,peer_group_id_x,peer_group_id_y
0,3,3
1,3,3
2,3,3
3,3,3
4,3,3


In [25]:
fact_ed_triage = fact_ed_triage.rename(
    columns={"peer_group_id_y": "peer_group_id"}
)

fact_ed_triage = fact_ed_triage.drop(columns=["peer_group_id_x"])

In [27]:
fact_ed_triage = fact_ed_triage.merge(
    dim_triage_category,
    on="triage_category",
    how="left"
)

fact_ed_triage.head()

,peer_group,triage_category,state,value,state_id,peer_group_id,triage_category_id
0,Principal referral and womens and children's h...,Resuscitation,NSW,99.9548,2,3,3
1,Principal referral and womens and children's h...,Emergency,NSW,74.9598,2,3,1
2,Principal referral and womens and children's h...,Urgent,NSW,67.904,2,3,5
3,Principal referral and womens and children's h...,Semi-urgent,NSW,73.1714,2,3,4
4,Principal referral and womens and children's h...,Non-urgent,NSW,86.9421,2,3,2


In [28]:
fact_ed_triage = fact_ed_triage[
    [
        "state_id",
        "peer_group_id",
        "triage_category_id",
        "value"
    ]
]

fact_ed_triage.head()

,state_id,peer_group_id,triage_category_id,value
0,2,3,3,99.9548
1,2,3,1,74.9598
2,2,3,5,67.904
3,2,3,4,73.1714
4,2,3,2,86.9421


In [29]:
fact_ed_triage.shape

(240, 4)